# STM32 Image Data Generator
Generates `image_data.c` and `image_data.h` for the STM32 binary classifier (digit 3 vs not 3).

**Output:** 20 test images — 10 threes, 10 non-threes — normalized and ready for X-CUBE-AI.

## Step 1 — Load and Prepare MNIST Data

In [ ]:
import numpy as np
from tensorflow.keras.datasets import mnist

# Load MNIST
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize to [0.0, 1.0]
x_test = x_test.astype('float32') / 255.0

print(f"Test set shape: {x_test.shape}")
print(f"Number of 3s in test set: {np.sum(y_test == 3)}")
print(f"Number of non-3s in test set: {np.sum(y_test != 3)}")

## Step 2 — Select 10 Threes and 10 Non-Threes

In [ ]:
import matplotlib.pyplot as plt

# Select images
threes_idx     = np.where(y_test == 3)[0][:10]
non_threes_idx = np.where(y_test != 3)[0][:10]
indices        = np.concatenate([threes_idx, non_threes_idx])
labels         = [1]*10 + [0]*10   # 1 = is a 3,  0 = is not a 3
images         = x_test[indices]   # shape: (20, 28, 28)
actual_digits  = y_test[indices]

# Preview all 20 images
fig, axes = plt.subplots(2, 10, figsize=(20, 4))
for i in range(20):
    ax = axes[i // 10][i % 10]
    ax.imshow(images[i], cmap='gray')
    ax.set_title(f"Digit:{actual_digits[i]}\nLabel:{labels[i]}", fontsize=8)
    ax.axis('off')
plt.suptitle('Top row: IS a 3 (label=1)    |    Bottom row: NOT a 3 (label=0)')
plt.tight_layout()
plt.show()

print("Top row (images 0-9):  all 3s")
print(f"Bottom row actual digits: {actual_digits[10:]}")

## Step 3 — Generate image_data.c

In [ ]:
with open("image_data.c", "w") as f:
    f.write("/*\n")
    f.write(" * image_data.c\n")
    f.write(" * Auto-generated for STM32 X-CUBE-AI binary classifier (digit 3)\n")
    f.write(" * 20 test images: 10 threes (label=1), 10 non-threes (label=0)\n")
    f.write(" */\n")
    f.write('#include "image_data.h"\n\n')

    # Labels array
    f.write("const int test_labels[NUM_TEST_IMAGES] = {\n    ")
    f.write(", ".join(str(l) for l in labels))
    f.write("\n};\n\n")

    # Images array
    f.write("const float test_images[NUM_TEST_IMAGES][IMAGE_SIZE] = {\n")
    for i, img in enumerate(images):
        digit = actual_digits[i]
        lbl   = labels[i]
        f.write(f"    /* Image {i:02d} - actual digit: {digit} - label: {lbl} ({'IS a 3' if lbl else 'NOT a 3'}) */\n")
        f.write("    {\n        ")
        vals = [f"{v:.6f}f" for v in img.flatten()]
        for j, v in enumerate(vals):
            f.write(v)
            if j < len(vals) - 1:
                f.write(", ")
                if (j + 1) % 10 == 0:
                    f.write("\n        ")
        comma = "," if i < 19 else ""
        f.write(f"\n    }}{comma}\n")
    f.write("};\n")

print("image_data.c generated successfully!")
print(f"  - {len(images)} images")
print(f"  - {sum(labels)} threes, {len(labels)-sum(labels)} non-threes")
print(f"  - Each image: {images[0].flatten().shape[0]} floats (28x28)")

## Step 4 — Generate image_data.h

In [ ]:
with open("image_data.h", "w") as f:
    f.write("/*\n")
    f.write(" * image_data.h\n")
    f.write(" * Auto-generated for STM32 X-CUBE-AI binary classifier (digit 3)\n")
    f.write(" */\n")
    f.write("#ifndef INC_IMAGE_DATA_H_\n")
    f.write("#define INC_IMAGE_DATA_H_\n\n")
    f.write("#define NUM_TEST_IMAGES  20\n")
    f.write("#define IMAGE_SIZE       784   /* 28 x 28 pixels */\n\n")
    f.write("/* 1 = IS a 3,  0 = NOT a 3 */\n")
    f.write("extern const int   test_labels[NUM_TEST_IMAGES];\n")
    f.write("extern const float test_images[NUM_TEST_IMAGES][IMAGE_SIZE];\n\n")
    f.write("#endif /* INC_IMAGE_DATA_H_ */\n")

print("image_data.h generated successfully!")

## Step 5 — Download Both Files

In [ ]:
from google.colab import files

files.download("image_data.c")
files.download("image_data.h")

print("Download started for both files.")
print("Replace both files in your STM32CubeIDE project and rebuild.")